# Inference using Python API

In this notebook, we will walk you through execution of an NLP (Task: MaskedLM) model on AIC100 device using Python API.

This notebook will introduce you to qaic python library which is used for inference on **AIC100**.

Let's dive right in and get started!

# Prerequisites

### Follow these steps to install qaic library.
```shell
pip install /opt/qti-aic/dev/lib/x86_64/qaic-0.0.1-py3-none-any.whl
pip install -r requirements.txt
```

### Follow these steps to setup and open this notebook. 

```shell
sudo /opt/qti-aic/dev/python/qaic-env/bin/pip install ipykernel
sudo jupyter kernelspec list
sudo /opt/qti-aic/dev/python/qaic-env/bin/python -m ipykernel install --user --name qaic-env --display-name "qaic-env"
sudo jupyter notebook --no-browser --port 8080 --allow-root
```


In [1]:
# Let's make sure the Python interpreter path is set properly.
import sys
sys.executable

'/opt/qti-aic/dev/python/qaic-env/bin/python'

# Overview 

Here is an overview of what we will cover in this demo.
1. **Setup**: We will begin by importing all the necessary libraries and importing all the required dependencies.
2. **torch-cpu inference**: Import the model, generate an input and run the model on CPU.
3. **ONNX conversion**: We will convert the pytorch model to onnx format. 
4. **Compilation**: Compile the model for QAIC100.
5. **Creating a Session and setting up inputs**: Create a qaic session and prepare the models for qaic runtime.
6. **Inference on AIC100**: Run the model with qaic api and decoding the output example.

# 1. Setup
Import the necessary libraries.

We will import the pre-trained model from Hugging Face ```transformers``` library. 


In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import sys
sys.path.append("/opt/qti-aic/examples/apps/qaic-python-sdk")
import qaic
import os
import torch
import onnx
from onnxsim import simplify
import argparse
import numpy as np
from typing import Dict, List, Optional, Tuple

/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose a model from ```transformers``` library 
For example: you can provide any of these pretrained models, but accordingly create ```<model_name>-config.yaml``` file containing compilation and execution options. 

```python
'distilbert-base-uncased' 'bert-base-uncased' 'bert-large-uncased' 'bert-base-cased' 'albert-base-v2'
```

In [3]:
model_card = 'bert-base-cased' # Provide a model name supported in transformers library.

In [4]:
# Import the pre-trained model
model = AutoModelForMaskedLM.from_pretrained(model_card)

# setup the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_card)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
# Sentence example
sentence = "The man worked as a [MASK]."

In [6]:
def get_example_input(tokenizer):
    max_length = 128
    encodings = tokenizer(sentence, max_length=max_length, truncation=True, padding="max_length", return_tensors='pt')
    inputIds = encodings["input_ids"]
    attentionMask = encodings["attention_mask"]
    mask_token_index = torch.where(encodings['input_ids'] == tokenizer.mask_token_id)[1]
    return inputIds, attentionMask, mask_token_index, encodings

In [7]:
# get input example 
inputIds, attentionMask, mask_token_index, inputs = get_example_input(tokenizer)

# 2. Inference using torch-cpu

In [8]:
# Execute on the host 
with torch.no_grad():
    pt_outputs = model(**inputs)

In [9]:
# decode the masked tokens from the pt_outputs variable.

mask_token_logits = pt_outputs.logits[0, mask_token_index, :]
top_5_results = torch.topk(mask_token_logits, 5, dim=1)
print("Model output (top5) from torch-cpu:")
for i in range(5):
    idx = top_5_results.indices[0].tolist()[i]
    val = top_5_results.values[0].tolist()[i]
    word = tokenizer.decode([idx])
    print(f"{i+1} :(word={word}, index={idx}, logit={round(val,2)})")

output_names = list(pt_outputs.keys())

Model output (top5) from torch-cpu:
1 :(word=lawyer, index=4545, logit=10.35)
2 :(word=waiter, index=17989, logit=10.1)
3 :(word=cop, index=9947, logit=10.04)
4 :(word=detective, index=9140, logit=9.92)
5 :(word=doctor, index=3995, logit=9.79)


# 3. ONNX conversion

In [10]:
# Setup dir for saving onnx and qpc files
gen_models_path = f"{model_card}/generatedModels"
os.makedirs(gen_models_path, exist_ok=True)
model_base_name = model_card


In [11]:
gen_models_path

'bert-base-cased/generatedModels'

In [12]:
# Set dynamic dims and axes.
dynamic_dims = {0: 'batch', 1 : 'sequence'}
dynamic_axes = {
    "input_ids" : dynamic_dims,
    "attention_mask" : dynamic_dims,
    "logits" : dynamic_dims
}
input_names = ["input_ids", "attention_mask"]
inputList = [inputIds, attentionMask]

model.eval() # setup the model in inference model.

torch.onnx.export(
    model,
    args=tuple(inputList),
    f=f"{gen_models_path}/{model_base_name}.onnx",
    verbose=False,
    input_names=input_names,
    output_names=["logits"],
    dynamic_axes=dynamic_axes,
    opset_version=11,
)
print("INFO: ONNX Model is being generated successfully")

# FIXME: if required apply onnxsimplify
# output_path_simp = output_path.replace(".onnx", "-simp.onnx")
# if not os.path.exists(output_path_simp):
# model_simp, check = simplify(output_path)
# onnx.save(model_simp, output_path)
# assert check, "Simplified ONNX model could not be validated."

# print("ONNX Model is being simplified and saved successfully.")

============= Diagnostic Run torch.onnx.export version 2.0.0+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

INFO: ONNX Model is being generated successfully


#### Modification 
Modify the onnx file to handle ```constants > FP16_Max and < FP16_Min ```

In [13]:
from onnx import numpy_helper
        
def fix_onnx_fp16(
    gen_models_path: str,
    model_base_name: str,
) -> str:
    finfo = np.finfo(np.float16)
    fp16_max = finfo.max
    fp16_min = finfo.min

    model = onnx.load(f"{gen_models_path}/{model_base_name}.onnx")
    fp16_fix = False
    for tensor in onnx.external_data_helper._get_all_tensors(model):
        nptensor = numpy_helper.to_array(tensor, gen_models_path)
        if nptensor.dtype == np.float32 and (
            np.any(nptensor > fp16_max) or np.any(nptensor < fp16_min)
        ):
            # print(f'tensor value : {nptensor} above {fp16_max} or below {fp16_min}')
            nptensor = np.clip(nptensor, fp16_min, fp16_max)
            new_tensor = numpy_helper.from_array(nptensor, tensor.name)
            tensor.CopyFrom(new_tensor)
            fp16_fix = True
            
    if fp16_fix:
        # Save FP16 model
        print("Found constants out of FP16 range, clipped to FP16 range")
        model_base_name += "_fix_outofrange_fp16"
        onnx.save(model, f=f"{gen_models_path}/{model_base_name}.onnx")
        print(f"Saving modified onnx file at {gen_models_path}/{model_base_name}.onnx")
    return model_base_name

fp16_model_name = fix_onnx_fp16(gen_models_path=gen_models_path, model_base_name=model_base_name)


Found constants out of FP16 range, clipped to FP16 range
Saving modified onnx file at bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16.onnx


# 4. Compilation step

In [14]:
# Function call to generate QPC binary by running qaic-exec command.
import os
import yaml
import inspect

def generate_bin(onnx_filename, yaml_filename):
    """
    Generate compiled binary for QAIC

    Args:
        onnx_path : path to onnx file.
        yaml_path : path to yaml file which has compile time arguments.

    Returns:
        qpc_path : path to qpc (compiled binary)
    """
    onnx_path = os.path.join(onnx_filename)
    yaml_path = os.path.join(yaml_filename)

    filename, extension = os.path.splitext(onnx_filename)
    onnx_folder = os.path.dirname(onnx_path)
    qpc_bin = os.path.join(filename+'_qpc')
    with open(yaml_path, "r") as file:
        yaml_data = yaml.load(file, Loader=yaml.FullLoader)

    if os.path.isdir(qpc_bin):
        print(f'INFO: Removing existing QPC {qpc_bin}')
        cmd = f'sudo rm -fr {qpc_bin}'
        os.system(cmd)
        print(f'INFO: Existing QPC {qpc_bin} is removed')

    # create the command string from the yaml arguments. 
    cmd_list = [f'/opt/qti-aic/exec/qaic-exec -m={onnx_path} -aic-hw -aic-hw-version={2.0}']

    # ignore the following arguments:
    ignore = ['num-activations']
    replace_dict = {'aic_num_cores':'aic-num-cores'}

    for arg, value in yaml_data.items():
        arg = arg.replace('_','-')
        if arg in ignore:
            continue
        if isinstance(value, bool):
            if value:# include the argument only if true; for example -convert-to-fp16
                cmd_list.append(f'-{arg}') 
        elif isinstance(value, dict):
            for subarg, subval in value.items():
                cmd_list.append(f'-{arg}={subarg},{subval}')
        else:
            cmd_list.append(f'-{arg}={value}')

    cmd_list.append(f'-aic-binary-dir={qpc_bin}')
    cmd_list.append(f'-compile-only')

    cmd = ' '.join(cmd_list)
    print(f'INFO: Running the compile cmd:')
    print(f'\n {cmd} \n')
    os.system(cmd)
    
    return qpc_bin

```generate_bin``` is a helper function which builds the ```qaic-exec``` CLI command based on the ```model-config.yaml``` file.

Same model-config.yaml is used for ```qaic.Session``` for running inference on AIC100 device. 

We are using ```bert-base-cased-config.yaml``` file for this model. It contains all the compile parameters used by helper function ```generate_bin```. ```generate_bin``` builds the ```qaic-exec``` CLI command to generate the QPC (Qualcomm Program Context) binary file. 

```bert-base-cased-config.yaml``` also contains inference parameters like num_activations which is used by ```qaic.Session``` along with input data for inference on the AIC100 device. 

In [15]:
# Lets print of the bert-base-cased-config.yaml 
options_path = f'{model_card}-config.yaml'
onnx_filePath = f'{model_card}/generatedModels/{fp16_model_name}.onnx'
_ = os.system(f'cat {options_path}')

# Compile parameters
aic_num_cores: 4
convert_to_fp16: true
onnx_define_symbol:
  batch: 1
  sequence: 128

# Inference Parameters
num_activations: 1


### Breakdown of yaml file parameters.
We have compiled the onnx file 
- with 4 NSP cores
- with float 16 precision
- defined onnx symbols

We will be running 1 activation of the network 
- num_activations: 1

In [16]:
# Compiling the onnx file for Qualcomm AI100

print("-----------------------------------------------------------------")
qpcPath = generate_bin(onnx_filename = onnx_filePath, yaml_filename=options_path)

-----------------------------------------------------------------
INFO: Running the compile cmd:

 /opt/qti-aic/exec/qaic-exec -m=bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16.onnx -aic-hw -aic-hw-version=2.0 -aic-num-cores=4 -convert-to-fp16 -onnx-define-symbol=batch,1 -onnx-define-symbol=sequence,128 -aic-binary-dir=bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16_qpc -compile-only 

Reading ONNX Model from bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16.onnx
Compile started ............... 
Compiling model with FP16 precision.
Generated binary is present at bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16_qpc


## NOTE:

There are three different approaches to invoke the device for inference. 

1. Utilizing a command line inferface (CLI) command - ```qaic-runner```
2. Employing ```qaic``` Python API (as shown below)
3. Leveraging the cpp api.

# 5. Creating a Session and setting up inputs

```qaic``` library is a set of APIs that provides support running inference on AIC100 backend. 

```Session```: Session is the entry point of these APIs. Session is a factory method which user needs to call to create an instance of session with AIC100 backend.

```Session(model_path: str, **kwargs)```

**Limitations**
- APIs are compatible with only python 3.8 
- These APIs are supported only on x86 platforms

In [17]:
# Compile and load the model
bert_sess = qaic.Session(model_path= qpcPath+'/programqpc.bin', options_path=options_path)

# alternatively, you can also provide arguments in the function call.
# bert_sess = qaic.Session(model_path= qpcPath+'/programqpc.bin', num_activations=2)

#### QAIC API


| **option** | **decription** |
| -------|------------|
| options_path (str) | Path to options yaml file |
| dev_id (int) | Device-id on which inference should run|
| num_activations (int)	      | Number of instances of network to be activated |
| set_size (int) 	      | Number of ExecObj to be created  |
 
 
 #### Function variables for session object
 
 | **API** | **Return Value** | **Example** | **Description** |
| -------|------------|---------|---------|
| run(input_dict) | A dict with output_name and output_value of inference	 | output = session.run(input_dict)	|input_dict should have input_name as key and value should be a numpy array.|
| reset()	| None | session.reset() | Releases all the device resources acquired by session |


In [18]:
input_shape, input_type = bert_sess.model_input_shape_dict['input_ids']
attn_shape, attn_type = bert_sess.model_input_shape_dict['attention_mask']
output_shape, output_type = bert_sess.model_output_shape_dict['logits']
print(f'Input token shape {input_shape} and type {input_type}')
print(f'Input attention mask shape {attn_shape} and type {attn_type}')
print(f'Output logits shape {output_shape} and type {output_type}')

Input token shape (1, 128) and type int64
Input attention mask shape (1, 128) and type int64
Output logits shape (1, 128, 28996) and type float32


# 6. Inference on AIC100

In [19]:
# Create a input dictionary for given input.
input_dict = {"input_ids": inputIds.numpy().astype(input_type), "attention_mask" : attentionMask.numpy().astype(attn_type)}

In [20]:
# Run the model on AIC100
output = bert_sess.run(input_dict)

qaic: WARNING: Acitvating network, this will add up to time of first inference


In [21]:
# Restructure the data from output buffer with output_shape, output_type
token_logits = np.frombuffer(output['logits'], dtype=output_type).reshape(output_shape)

# Decode the output.
mask_token_logits = torch.from_numpy(token_logits[0, mask_token_index, :]).unsqueeze(0)
top_5_results = torch.topk(mask_token_logits, 5, dim=1)
print("Model output (top5) from AIC100:")
for i in range(5):
    idx = top_5_results.indices[0].tolist()[i]
    val = top_5_results.values[0].tolist()[i]
    word = tokenizer.decode([idx])
    print(f"{i+1} :(word={word}, index={idx}, logit={round(val,2)})")


Model output (top5) from AIC100:
1 :(word=lawyer, index=4545, logit=10.34)
2 :(word=waiter, index=17989, logit=10.09)
3 :(word=cop, index=9947, logit=10.04)
4 :(word=detective, index=9140, logit=9.91)
5 :(word=doctor, index=3995, logit=9.78)


# Sample config file

```yaml
# Number of aic cores to be used for inference 
aic_num_cores: 4


# Number of instances of network to be activated.
num_activations: 1


# Run all floating-point in fp16
convert_to_fp16: true

#Define an onnx symbol with its value
onnx_define_symbol:
  batch: 1

# Stores model binaries at directory location provided
output_dir: './resnet_qpc'

# Device-id on which inference should run
dev_id: 0

# Number of ExecObj to be created
set_size: 4 

#Effort level to reduce the on-chip memory
mos: 2

# Factor to increasing splitting of network for parallelism
ols: 3

# Output node names should be in the order as present in model file. This option is mandatory for TF models
output_node_names:
- 'network/cond/last_fc_z'


# Provide input node name with its data type and shape. Dict must contain keys 'input_name', 'input_type','input_shape'.  This is mandatory for pytorch models
model_inputs:
- input_name: 'input_img'
  input_type: 'float32'
  input_shape: 
  - 1
  - 3
  - 224
  - 224
```

In [22]:
## Questions FIXME
1. is set size compile or runtime?
2. need to check if model preparator provides any advantage in terms of model size or inference speed.
3. Any diagrams required?

SyntaxError: invalid syntax (1399330289.py, line 3)